# TechStore Plus — LangChain LCEL Customer Service Agent

**M1 Capstone Part 2 of 3 — Week 2** | Refactoring the Week 1 OpenAI chatbot into a composable, observable LangChain pipeline.

## Architecture Overview (Approach B — Intermediate)

| Step | Component | Key Technique | Output |
|------|-----------|---------------|--------|
| 1 | Query Analysis & Classification | `with_structured_output(QueryAnalysis)` | Pydantic model |
| 2 | Dynamic Response Generation | `RunnableLambda` router + category-specific prompts | `AIMessage` |
| 3 | Conversation Summary & Persistence | Structured mapping → `ConversationSummary` | Pydantic model |

All three components are linked via **LCEL pipe operators** into a single chain:
```
RunnablePassthrough.assign(analysis=...) | RunnablePassthrough.assign(response=...) | RunnableLambda(build_summary)
```

**Why Approach B:** The `RunnableLambda` router cleanly encapsulates category dispatch (one place to change routing logic). The summary is built deterministically from the structured analysis, avoiding an extra LLM call.

## 1. Project Setup & LangSmith Configuration

LangSmith tracing is enabled via environment variables that **must be set before** any LangChain LLM is instantiated — LangChain reads these at creation time, not invocation time.

To enable tracing:
1. Get your API key at `smith.langchain.com`
2. Add to `.env`: `LANGCHAIN_API_KEY=ls__...`
3. Traces will appear in the `Advanced-Customer-Agent` project in the LangSmith UI

In [1]:
import os
import json
import uuid
from datetime import datetime
from pathlib import Path
from typing import List, Optional, Literal

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from pydantic import BaseModel, Field

# LangSmith tracing must be configured before any LLM is instantiated.
# Keep tracing disabled when LANGCHAIN_API_KEY is empty to avoid unauthorized trace uploads.
os.environ.setdefault('LANGCHAIN_PROJECT', 'Advanced-Customer-Agent')
if os.getenv('LANGCHAIN_API_KEY', '').strip():
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    print('LangSmith tracing ENABLED — project: ' + os.environ['LANGCHAIN_PROJECT'])
else:
    os.environ['LANGCHAIN_TRACING_V2'] = 'false'
    print('LangSmith tracing DISABLED — add LANGCHAIN_API_KEY=ls__... to .env to enable.')
    print('Chains will run normally but traces will not be uploaded.')

DATA_DIR = Path('conversation_data')
DATA_DIR.mkdir(exist_ok=True)

LangSmith tracing ENABLED — project: Advanced-Customer-Agent


## 2. Company Context — TechStore Plus

The same knowledge base from Week 1 is reused here. It is injected into the response-generation prompts in Component 2 to ground the LLM's answers in real company policies — preventing hallucinated policies.

In [2]:
COMPANY_CONTEXT = {
    'company_name': 'TechStore Plus',
    'description': 'Your Trusted Technology Store',
    'sector': 'E-commerce for technology products',
    'location': 'New York, USA',
    'business_hours': {'monday_friday': '09:00-18:00', 'saturday': '10:00-14:00'},
    'key_services': [
        'Sales', 'Technical Support', 'Warranty', 'Financing', 'Trade-ins'
    ],
    'products': [
        'Laptops', 'Desktop PCs', 'Workstations', 'Smartphones', 'Tablets',
        'Headphones', 'Mice', 'Keyboards', 'Gaming consoles', 'Cameras'
    ],
    'policies': {
        'shipping': 'Free nationwide shipping for purchases over $500.',
        'returns': '30 days for exchanges and 7 days for refunds.',
        'warranty': '12 months warranty on all products.',
        'installation': 'Home technical service is available.',
        'extended_warranty': 'Optional extended warranty for 1 additional year.',
        'financing': 'Interest-free installments and payment plans are available.',
        'trade_ins': 'Eligible devices may be evaluated for trade-in credit toward a new purchase.'
    },
    'contact': {
        'email': 'support@techstoreplus.com',
        'phone': '1-800-TECH-PLUS',
        'chat': 'Available on website 24/7'
    }
}

COMPANY_CONTEXT_STR = json.dumps(COMPANY_CONTEXT, indent=2)
print('Company context loaded:', COMPANY_CONTEXT['company_name'])

Company context loaded: TechStore Plus


## 3. Component 1: Query Analysis & Classification

### Design Choices

**`with_structured_output(QueryAnalysis)`** — Instead of asking the LLM to return JSON and parsing it manually (as in Week 1), we use LangChain's structured output method. It routes through OpenAI's function-calling API, which guarantees the output matches the Pydantic schema. Pydantic validation runs automatically — if the model returns an invalid category string, the call raises a `ValidationError` instead of silently passing bad data downstream.

**`temperature=0`** — Classification should be deterministic. Zero temperature removes randomness from category and urgency assignments.

**LCEL pipe:** `analysis_prompt | analysis_llm.with_structured_output(QueryAnalysis)` — the prompt formats the raw query into a message list; the structured LLM returns a validated `QueryAnalysis` instance.

In [3]:
# --- Pydantic models for structured output ---

class ExtractedEntities(BaseModel):
    """Key entities extracted from the customer query."""
    product_name: Optional[str] = Field(None, description='The specific product mentioned by the user')
    order_number: Optional[str] = Field(None, description='The order number mentioned (e.g. #TEC-2024-001)')
    date: Optional[str] = Field(None, description='Any date mentioned by the user (e.g. December 15th)')


class QueryAnalysis(BaseModel):
    """Analyzes and classifies a customer service query."""
    query_category: Literal['technical_support', 'billing', 'returns', 'product_inquiry', 'general_information']
    urgency_level: Literal['low', 'medium', 'high']
    customer_sentiment: Literal['positive', 'neutral', 'negative']
    entities: ExtractedEntities


# temperature=0 for deterministic, consistent classification
analysis_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Minimal system prompt avoids biasing the classification toward any particular category.
# The human message contains only the raw query — no category hints — to test true understanding.
analysis_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are an expert customer service query classifier for TechStore Plus.\n'
     'Analyze the customer query and extract structured information. Be precise and consistent.\n\n'
     'Categories: technical_support | billing | returns | product_inquiry | general_information\n'
     'Urgency: low (informational) | medium (needs resolution) | high (emergency/time-critical)\n'
     'Sentiment: positive | neutral | negative'),
    ('human', 'Customer query: {query}')
])

# Component 1 chain — the pipe operator connects the prompt to the structured LLM.
# with_structured_output() uses function-calling to enforce the Pydantic schema automatically.
analysis_chain = analysis_prompt | analysis_llm.with_structured_output(QueryAnalysis)

print('Component 1 ready: analysis_chain')
print('Output type:', QueryAnalysis.__name__)

Component 1 ready: analysis_chain
Output type: QueryAnalysis


In [4]:
# Standalone test of Component 1 — verifies schema enforcement and entity extraction
test_result = analysis_chain.invoke({'query': 'This is an emergency! My order #TEC-2024-001 never arrived!'})
print('query_category: ', test_result.query_category)
print('urgency_level:  ', test_result.urgency_level)
print('customer_sentiment:', test_result.customer_sentiment)
print('entities:       ', test_result.entities.model_dump())

query_category:  returns
urgency_level:   high
customer_sentiment: negative
entities:        {'product_name': None, 'order_number': 'TEC-2024-001', 'date': None}


## 4. Component 2: Dynamic Response Generation

### Design Choices

**Why different prompts per category?** A technical issue requires troubleshooting steps; a billing inquiry requires document-retrieval steps; a return needs policy explanation. One generic prompt would produce weaker, less actionable responses. Category-specific prompts let each specialist persona shine.

**Why `RunnableLambda` for routing?** LCEL's pipe operator (`|`) is static — it cannot branch at runtime based on data. Wrapping the routing logic in a `RunnableLambda` lets us inspect `analysis.query_category` at invocation time and select the correct `ChatPromptTemplate` dynamically. This keeps all routing logic in one function — no hardcoded category strings scattered across the chain.

**`temperature=0.4`** — Response generation benefits from some variability to sound natural, while staying grounded in policy facts.

In [5]:
# temperature=0.4 for natural, slightly varied responses while staying policy-accurate
response_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.4)

# Each category has a purpose-built system prompt defining the specialist persona
# and the specific response guidelines for that inquiry type.
CATEGORY_PROMPTS = {
    'technical_support': ChatPromptTemplate.from_messages([
        ('system',
         'You are a TechStore Plus technical support specialist.\n'
         'Company context: {company_context}\n\n'
         'Guidelines:\n'
         '- Begin with empathy if sentiment is negative or frustrated.\n'
         '- Provide 2-3 concrete, actionable troubleshooting steps.\n'
         '- If urgency is high, offer immediate escalation to a senior technician.\n'
         '- Reference the specific product if mentioned.\n'
         '- End with a clear next step the customer should take.'),
        ('human',
         'Customer query: {query}\n'
         'Sentiment: {sentiment} | Urgency: {urgency} | Entities: {entities_json}\n\n'
         'Provide a helpful, empathetic technical support response with clear next steps.')
    ]),
    'billing': ChatPromptTemplate.from_messages([
        ('system',
         'You are a TechStore Plus billing specialist.\n'
         'Company context: {company_context}\n\n'
         'Guidelines:\n'
         '- Be professional and formal in tone.\n'
         '- Reference the order number if mentioned.\n'
         '- Explain the exact document retrieval or payment resolution process.\n'
         '- Offer to resend receipts or payment confirmations via email.\n'
         '- End with a specific action the customer or the billing team will take.'),
        ('human',
         'Customer query: {query}\n'
         'Sentiment: {sentiment} | Urgency: {urgency} | Entities: {entities_json}\n\n'
         'Provide a clear, professional billing support response with specific next steps.')
    ]),
    'returns': ChatPromptTemplate.from_messages([
        ('system',
         'You are a TechStore Plus returns specialist.\n'
         'Company context: {company_context}\n\n'
         'Guidelines:\n'
         '- Clearly state the return windows: 30 days for exchanges, 7 days for refunds.\n'
         '- Walk through the return process step by step.\n'
         '- If urgency is high, fast-track the request and offer priority handling.\n'
         '- Acknowledge any inconvenience with genuine empathy.\n'
         '- End with the immediate next step (e.g. initiate return online or visit store).'),
        ('human',
         'Customer query: {query}\n'
         'Sentiment: {sentiment} | Urgency: {urgency} | Entities: {entities_json}\n\n'
         'Provide a clear, empathetic returns/refund response with step-by-step instructions.')
    ]),
    'product_inquiry': ChatPromptTemplate.from_messages([
        ('system',
         'You are a TechStore Plus sales advisor.\n'
         'Company context: {company_context}\n\n'
         'Guidelines:\n'
         '- Be enthusiastic and helpful — match the customer energy for positive sentiment.\n'
         '- Mention product availability, pricing range, and shipping policies.\n'
         '- Offer relevant recommendations if a budget or product category is mentioned.\n'
         '- Highlight the free shipping policy for purchases over $500.\n'
         '- End with an invitation to complete the purchase or visit the store.'),
        ('human',
         'Customer query: {query}\n'
         'Sentiment: {sentiment} | Urgency: {urgency} | Entities: {entities_json}\n\n'
         'Provide an informative, sales-friendly product inquiry response.')
    ]),
    'general_information': ChatPromptTemplate.from_messages([
        ('system',
         'You are a TechStore Plus customer service representative.\n'
         'Company context: {company_context}\n\n'
         'Guidelines:\n'
         '- Be friendly and comprehensive.\n'
         '- Match your tone to the customer sentiment.\n'
         '- Provide relevant company information and direct to the right department if needed.\n'
         '- End with clear contact information or next steps.'),
        ('human',
         'Customer query: {query}\n'
         'Sentiment: {sentiment} | Urgency: {urgency} | Entities: {entities_json}\n\n'
         'Provide a helpful general information response.')
    ]),
}


def route_response(inputs: dict):
    """
    Routes the query to the correct category-specific prompt and generates a response.

    Why RunnableLambda: The routing decision depends on analysis.query_category, which is
    only known at runtime after Component 1 runs. LCEL pipes are static — they cannot
    branch conditionally. Wrapping this function in RunnableLambda makes it a first-class
    LCEL citizen while keeping all routing logic in one place.

    Trade-off vs Approach A: Approach A would call chain.invoke() separately per category
    in the caller code. Here, the dispatch is fully encapsulated — the chain caller never
    needs to know which category was selected.
    """
    analysis: QueryAnalysis = inputs['analysis']
    query: str = inputs['query']

    # Select the category-specific prompt; fall back to general_information if unknown
    prompt = CATEGORY_PROMPTS.get(analysis.query_category, CATEGORY_PROMPTS['general_information'])

    # Build and invoke a mini-chain: selected prompt | response LLM
    chain = prompt | response_llm
    return chain.invoke({
        'query': query,
        'sentiment': analysis.customer_sentiment,
        'urgency': analysis.urgency_level,
        'entities_json': analysis.entities.model_dump_json(),
        'company_context': COMPANY_CONTEXT_STR,
    })


print('Component 2 ready: route_response (to be wrapped in RunnableLambda)')
print('Categories covered:', list(CATEGORY_PROMPTS.keys()))

Component 2 ready: route_response (to be wrapped in RunnableLambda)
Categories covered: ['technical_support', 'billing', 'returns', 'product_inquiry', 'general_information']


## 5. Component 3: Conversation Summarization & Persistence

### Design Choices

**Deterministic summary assembly:** The `ConversationSummary` is built programmatically from the already-computed `QueryAnalysis` fields rather than making an additional LLM call. This keeps Component 3 deterministic (no randomness in summary fields), faster (one fewer API call), and cheaper (no extra tokens). The `conversation_summary` sentence is templated from structured fields — ensuring it always contains accurate category/sentiment/urgency data.

**Urgency → Resolution mapping:** `high` → `escalated`, `medium` → `pending`, `low` → `resolved`. This mirrors the business logic from Week 1 and is determined by the structured analysis, not guessed by the LLM.

**Persistence:** Each conversation is saved as an individual JSON file (same structure as Week 1) and then merged by `consolidate_conversations()`.

In [6]:
class ConversationSummary(BaseModel):
    """A structured summary of the customer service interaction."""
    timestamp: str
    customer_id: str = 'auto_generated'
    conversation_summary: str = Field(description='A concise, one-sentence summary of the interaction.')
    query_category: str
    customer_sentiment: str
    urgency_level: str
    mentioned_products: List[str]
    extracted_information: dict
    resolution_status: Literal['resolved', 'pending', 'escalated']
    actions_taken: List[str] = Field(description='A list of actions the agent took or suggested.')
    follow_up_required: bool


def _response_text(response) -> str:
    """Extracts readable text from a LangChain message or plain string."""
    content = getattr(response, 'content', response)
    if isinstance(content, list):
        return ' '.join(str(part) for part in content)
    return str(content or '')


def _short_action_from_response(response) -> str:
    """Keeps the final summary connected to the actual generated response."""
    text = _response_text(response).replace('\n', ' ').strip()
    if not text:
        return 'Generated customer response with clear next steps'
    return 'Generated customer response: ' + text[:180] + ('...' if len(text) > 180 else '')


def build_summary(inputs: dict) -> ConversationSummary:
    """
    Assembles ConversationSummary from Component 1 analysis + Component 2 response.

    Design choice: The categorical fields come from the validated QueryAnalysis object,
    while actions_taken records the actual generated response so Component 3 is populated
    from information gathered across the whole chain.
    """
    analysis: QueryAnalysis = inputs['analysis']
    response = inputs.get('response')
    customer_id: str = inputs.get('customer_id', 'CUST-' + uuid.uuid4().hex[:8].upper())

    # Extract products and structured info from entities
    mentioned_products = []
    if analysis.entities.product_name:
        mentioned_products.append(analysis.entities.product_name)

    extracted_information = {}
    if analysis.entities.order_number:
        extracted_information['order_number'] = analysis.entities.order_number
    if analysis.entities.date:
        extracted_information['date'] = analysis.entities.date

    # Map urgency to resolution status (business logic, not LLM guessing)
    urgency_to_status = {'high': 'escalated', 'medium': 'pending', 'low': 'resolved'}
    resolution_status = urgency_to_status.get(analysis.urgency_level, 'pending')

    # Build a one-sentence summary from structured fields
    entity_detail = ''
    if analysis.entities.order_number:
        entity_detail = ' regarding order ' + analysis.entities.order_number
    elif analysis.entities.product_name:
        entity_detail = ' regarding ' + analysis.entities.product_name

    summary_text = (
        'Customer contacted TechStore Plus with a '
        + analysis.query_category.replace('_', ' ') + ' inquiry'
        + entity_detail + '; sentiment was ' + analysis.customer_sentiment
        + ' with ' + analysis.urgency_level + ' urgency, case '
        + resolution_status + '.'
    )

    actions_taken = [
        'Classified query as ' + analysis.query_category.replace('_', ' '),
        _short_action_from_response(response),
        'Routed customer to appropriate support channel',
    ]

    return ConversationSummary(
        timestamp=datetime.now().isoformat(timespec='seconds'),
        customer_id=customer_id,
        conversation_summary=summary_text,
        query_category=analysis.query_category,
        customer_sentiment=analysis.customer_sentiment,
        urgency_level=analysis.urgency_level,
        mentioned_products=mentioned_products,
        extracted_information=extracted_information,
        resolution_status=resolution_status,
        actions_taken=actions_taken,
        follow_up_required=analysis.urgency_level in ('high', 'medium'),
    )


def save_conversation_json(summary: ConversationSummary) -> Path:
    """Saves the ConversationSummary as an individual JSON file with timestamp."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    file_path = DATA_DIR / ('week2_conversation_' + summary.customer_id + '_' + timestamp + '.json')
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(summary.model_dump(), f, indent=2, ensure_ascii=False)
    return file_path


def consolidate_conversations(output_file: str = 'week2_consolidated_conversations.json') -> Path:
    """Merges only current Week 2 LCEL conversation JSON files into one consolidated file."""
    all_conversations = []
    for file_path in sorted(DATA_DIR.glob('week2_conversation_*.json')):
        with open(file_path, 'r', encoding='utf-8') as f:
            all_conversations.append(json.load(f))
    output_path = DATA_DIR / output_file
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump({
            'generated_at': datetime.now().isoformat(timespec='seconds'),
            'total_conversations': len(all_conversations),
            'conversations': all_conversations,
        }, f, indent=2, ensure_ascii=False)
    return output_path


print('Component 3 ready: ConversationSummary, build_summary, save_conversation_json')

Component 3 ready: ConversationSummary, build_summary, save_conversation_json


## 6. Full LCEL Chain Assembly

### How the pipe works

```
Input: {"query": str, "customer_id": str}
         ↓
RunnablePassthrough.assign(analysis=...)   → adds "analysis": QueryAnalysis  (Component 1)
         ↓
RunnablePassthrough.assign(response=...)   → adds "response": AIMessage       (Component 2)
         ↓
RunnableLambda(build_summary)              → returns ConversationSummary       (Component 3)
```

**`RunnablePassthrough.assign(key=runnable)`** adds a new key to the input dict while passing all existing keys through unchanged. This is how we accumulate state across components without losing previous results.

**Why wrap the lambda in `RunnableLambda`?** `RunnablePassthrough.assign()` accepts either a `Runnable` or a plain callable. Explicit `RunnableLambda` wrapping ensures LangSmith traces the step as a named node in the chain graph, making the trace more readable.

In [7]:
# --- Full LCEL Chain ---
#
# Step 1: RunnableLambda extracts {"query"} from the full input to pass to analysis_chain,
#         which expects only {"query": str}. assign() adds "analysis" to the running dict.
#
# Step 2: RunnableLambda(route_response) receives the full dict (with "analysis" now present)
#         and routes to the correct category prompt. assign() adds "response" to the dict.
#
# Step 3: RunnableLambda(build_summary) receives the full dict and maps it to ConversationSummary.

chain_with_context = (
    RunnablePassthrough.assign(
        analysis=RunnableLambda(lambda x: {'query': x['query']}) | analysis_chain
    )
    | RunnablePassthrough.assign(
        response=RunnableLambda(route_response)
    )
    | RunnablePassthrough.assign(
        summary=RunnableLambda(build_summary)
    )
)

# The PDF asks for the final chain output to be a ConversationSummary instance.
# chain_with_context is used only in the examples so we can also display the customer response.
full_chain = chain_with_context | RunnableLambda(lambda x: x['summary'])

print('Full chain assembled with LCEL pipe operators:')
print('  RunnablePassthrough.assign(analysis=lambda | analysis_chain)')
print('  | RunnablePassthrough.assign(response=RunnableLambda(route_response))')
print('  | RunnablePassthrough.assign(summary=RunnableLambda(build_summary))')
print('  | RunnableLambda(lambda x: x["summary"])')
print()
print('Input:  {"query": str, "customer_id": str (optional)}')
print('Output:', ConversationSummary.__name__)
print('Demo chain output keys: query, customer_id, analysis, response, summary')

Full chain assembled with LCEL pipe operators:
  RunnablePassthrough.assign(analysis=lambda | analysis_chain)
  | RunnablePassthrough.assign(response=RunnableLambda(route_response))
  | RunnablePassthrough.assign(summary=RunnableLambda(build_summary))
  | RunnableLambda(lambda x: x["summary"])

Input:  {"query": str, "customer_id": str (optional)}
Output: ConversationSummary
Demo chain output keys: query, customer_id, analysis, response, summary


## 7. Usage Example — Single Query

Running the "Urgent-Negative" test case end-to-end through all three components.

In [8]:
test_query = 'This is an emergency! My order #TEC-2024-001 never arrived and I need that laptop for work tomorrow!'
customer_id = 'CUST-' + uuid.uuid4().hex[:8].upper()

run = chain_with_context.invoke({
    'query': test_query,
    'customer_id': customer_id,
})
result: ConversationSummary = run['summary']

print('=' * 70)
print('QUERY:', test_query)
print('=' * 70)
print('customer_id:      ', result.customer_id)
print('query_category:   ', result.query_category)
print('urgency_level:    ', result.urgency_level)
print('customer_sentiment:', result.customer_sentiment)
print('resolution_status:', result.resolution_status)
print('mentioned_products:', result.mentioned_products)
print('extracted_info:   ', result.extracted_information)
print('follow_up:        ', result.follow_up_required)
print()
print('CUSTOMER RESPONSE:')
print(_response_text(run['response']))
print()
print('CONVERSATION SUMMARY:')
print(result.conversation_summary)
print()
print('ACTIONS TAKEN:')
for action in result.actions_taken:
    print(' -', action)
print()
print('FULL JSON:')
print(json.dumps(result.model_dump(), indent=2))

QUERY: This is an emergency! My order #TEC-2024-001 never arrived and I need that laptop for work tomorrow!
customer_id:       CUST-0E176B1D
query_category:    returns
urgency_level:     high
customer_sentiment: negative
resolution_status: escalated
mentioned_products: ['laptop']
extracted_info:    {'order_number': '#TEC-2024-001', 'date': 'tomorrow'}
follow_up:         True

CUSTOMER RESPONSE:
I’m truly sorry to hear that your laptop order #TEC-2024-001 hasn’t arrived, especially with your work deadline tomorrow. I understand how crucial this is for you, and I appreciate your patience in this situation.

To address this urgently, let’s take the following steps:

1. **Check Order Status**: First, I recommend checking the tracking information for your order. You can do this by logging into your TechStore Plus account or by clicking on the tracking link provided in your order confirmation email.

2. **Contact Shipping Provider**: If the tracking indicates that the package is delayed, ple

## 8. All 6 Suggested Test Queries

Running the full chain against all six test cases from the spec. Each covers a different category, sentiment, and urgency combination to validate end-to-end routing and classification.

In [9]:
TEST_QUERIES = [
    {
        'label': 'Neutral-Informative',
        'query': 'Hello, I\'d like to know if you have the new iPhone 15 in stock and how much shipping costs to Chicago'
    },
    {
        'label': 'Urgent-Negative',
        'query': 'This is an emergency! My order #TEC-2024-001 never arrived and I need that laptop for work tomorrow!'
    },
    {
        'label': 'Satisfied-Positive',
        'query': 'Thank you so much for the excellent service with my previous purchase, I want to buy gaming headphones'
    },
    {
        'label': 'Frustrated-Technical',
        'query': 'I can\'t configure the router I bought last week, I\'ve tried everything and it doesn\'t work'
    },
    {
        'label': 'Formal-Billing',
        'query': 'Good morning, I need the receipt for my purchase from December 15th, order #TEC-2023-089'
    },
    {
        'label': 'Warranty-Query',
        'query': 'I bought a tablet 8 months ago and now it won\'t turn on, how do I use the warranty?'
    },
]

results = []

for item in TEST_QUERIES:
    customer_id = 'CUST-' + uuid.uuid4().hex[:8].upper()
    run = chain_with_context.invoke({
        'query': item['query'],
        'customer_id': customer_id,
    })
    result: ConversationSummary = run['summary']
    results.append({
        'label': item['label'],
        'query': item['query'],
        'analysis': run['analysis'],
        'response': run['response'],
        'result': result,
    })

    print('[' + item['label'] + ']')
    print('  query_category:    ', result.query_category)
    print('  urgency_level:     ', result.urgency_level)
    print('  customer_sentiment:', result.customer_sentiment)
    print('  resolution_status: ', result.resolution_status)
    print('  mentioned_products:', result.mentioned_products)
    print('  extracted_info:    ', result.extracted_information)
    print('  customer_response: ', _response_text(run['response'])[:240].replace('\n', ' ') + '...')
    print('  follow_up_required:', result.follow_up_required)
    print()

print('All', len(results), 'test queries completed.')

[Neutral-Informative]
  query_category:     product_inquiry
  urgency_level:      low
  customer_sentiment: neutral
  resolution_status:  resolved
  mentioned_products: ['iPhone 15']
  extracted_info:     {}
  customer_response:  Hello! Thank you for reaching out to us at TechStore Plus! 🎉  I'm excited to let you know that we do have the new iPhone 15 in stock! The pricing typically starts around $999, depending on the model and storage capacity you choose.   As for...
  follow_up_required: False

[Urgent-Negative]
  query_category:     returns
  urgency_level:      high
  customer_sentiment: negative
  resolution_status:  escalated
  mentioned_products: ['laptop']
  extracted_info:     {'order_number': '#TEC-2024-001', 'date': 'tomorrow'}
  customer_response:  I'm truly sorry to hear about the inconvenience you're experiencing with your order #TEC-2024-001. I understand how important it is for you to have your laptop for work tomorrow, and I want to help you resolve this as quickly as

## 9. Save Summaries & Consolidate

Each conversation summary is saved as an individual JSON file (same structure as Week 1) and then merged into a single consolidated file for downstream analysis.

In [10]:
saved_files = []
for item in results:
    file_path = save_conversation_json(item['result'])
    saved_files.append(file_path)
    print('Saved:', file_path)

consolidated = consolidate_conversations()
print('\nConsolidated file:', consolidated)
print('Total current Week 2 conversations saved:', len(saved_files))

Saved: conversation_data\week2_conversation_CUST-AAD501B7_20260526_123940.json
Saved: conversation_data\week2_conversation_CUST-D4ADCB91_20260526_123940.json
Saved: conversation_data\week2_conversation_CUST-CDC44E51_20260526_123940.json
Saved: conversation_data\week2_conversation_CUST-8C7AFA2B_20260526_123940.json
Saved: conversation_data\week2_conversation_CUST-9D867BA5_20260526_123940.json
Saved: conversation_data\week2_conversation_CUST-D5EF3056_20260526_123940.json

Consolidated file: conversation_data\week2_consolidated_conversations.json
Total current Week 2 conversations saved: 6


In [ ]:
EXPECTED_RESULTS = {
    'Neutral-Informative': ('product_inquiry', 'low', 'neutral'),
    'Urgent-Negative': ('general_information or product_inquiry', 'high', 'negative'),
    'Satisfied-Positive': ('product_inquiry', 'low', 'positive'),
    'Frustrated-Technical': ('technical_support', 'medium/high', 'negative'),
    'Formal-Billing': ('billing', 'medium', 'neutral'),
    'Warranty-Query': ('general_information or technical_support', 'medium', 'neutral'),
}

print('| Label | Expected Category | Expected Urgency | Expected Sentiment | Actual Category | Actual Urgency | Actual Sentiment |')
print('|-------|-------------------|------------------|--------------------|-----------------|----------------|------------------|')
for item in results:
    expected_category, expected_urgency, expected_sentiment = EXPECTED_RESULTS[item['label']]
    summary = item['result']
    print(
        '| ' + item['label']
        + ' | `' + expected_category + '`'
        + ' | `' + expected_urgency + '`'
        + ' | `' + expected_sentiment + '`'
        + ' | `' + summary.query_category + '`'
        + ' | `' + summary.urgency_level + '`'
        + ' | `' + summary.customer_sentiment + '` |'
    )

## 10. Results Analysis

### Classification Accuracy

The code cell above generates the expected-vs-actual classification table directly from the `results` list produced by the six suggested test queries.

### Key Observations

**Component 1 (Classification):**
- `with_structured_output()` eliminates manual JSON parsing errors because Pydantic validation catches schema violations before they propagate downstream.
- The "Urgent-Negative" delivery case may classify as `general_information` because the required Week 2 schema does not include a delivery/shipping issue category.
- Entity extraction should pick up `#TEC-2024-001`, `#TEC-2023-089`, product names, and dates where present.

**Component 2 (Routing):**
- The `RunnableLambda` router dispatches each query to its category-specific prompt.
- The demo cells now print the generated customer response, making it possible to verify sentiment matching, entity acknowledgement, and actionable next steps.
- Routing is encapsulated in `route_response()`, so category dispatch has one central maintenance point.

**Component 3 (Summary):**
- `ConversationSummary` is still the final output of `full_chain`, as required by the PDF.
- `actions_taken` now records a short excerpt from the actual generated response, so the summary is populated from both the analysis and response-generation stages.
- Week 2 persistence writes `week2_conversation_*.json` and `week2_consolidated_conversations.json`, avoiding stale Week 1 files that used different categories.

**LangSmith evidence:**
- Add a screenshot or public trace link here after running the "Urgent-Negative" query.
- Project name: `Advanced-Customer-Agent`.
- Suggested trace to capture: the `chain_with_context.invoke()` run from the single-query example, because it includes analysis, routed response generation, and summary creation.

### Week 2 → Week 3 Bridge

Week 3 (graded) extends this chain with:
- **HybridMemory** — capped message buffer + rolling summary for multi-turn conversations
- **MemoryAgent** — tool-using agent with `@tool` decorators and `create_agent`
- (Bonus) MCP server integration

The LCEL chain built here serves as the foundation: `analysis_chain` feeds into the agent's tool calls, and `ConversationSummary` becomes the agent's memory log format.